# DeepMeow — Computer Vision Research Notebook (Google Colab)

Welcome to the experiment workspace for **DeepMeow**, a research project focused on building a single-shot object detector and multi-object tracker from first principles in PyTorch.

### Notebook Sections:
- **Cells 1-8 (Week 1)**: Environment setup, dataset download, backbone verification
- **Cells 9-15 (Week 2)**: FPN, detection head, loss functions, full detector

> **Note for Collaborators**: Ensure your Colab runtime accelerator is set to GPU (`Runtime -> Change runtime type -> T4 GPU`).

## 1. Hardware Initialization & GPU Verification

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU available: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
else:
    print('WARNING: No GPU detected. Please switch runtime to GPU (Runtime -> Change runtime type -> T4 GPU).')

## 2. Environment Setup & Repository Synchronization

In [ ]:
import os

REPO_URL = 'https://github.com/IliyaJz/DeepMeow.git'
REPO_DIR = '/content/DeepMeow'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Pulling latest changes...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'\nWorking directory: {os.getcwd()}')

In [ ]:
print('Installing dependencies...')
!pip install -q -r requirements.txt
print('All packages installed!')

## 3. Persistent Storage Setup (Google Drive)

Mounting Google Drive allows us to persist downloaded datasets and training checkpoints across Colab session disconnects.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted!')
    print('Data will be saved to Google Drive.')
else:
    print('Skipping Drive mount. Data will be stored in Colab /content/ (resets each session).')

## 4. Dataset Acquisition & COCO Class Filtering

We execute `downloader.py` which filters COCO 2017 for cat annotations (`category_id == 17`)
and downloads ~3,000 training + ~500 validation images.

In [ ]:
!python src/data/downloader.py

## 5. Dataset Validation & Integrity Check

We inspect the generated JSON annotation files and verify that the downloaded `.jpg` image files match our annotation records.

In [ ]:
import json
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/DeepMeow/data')
local_data = Path('data')
data_root  = drive_data if (USE_DRIVE and (drive_data / 'annotations/train.json').exists()) else local_data
print(f'Using dataset root: {data_root.resolve()}\n')

for split in ['train', 'val']:
    ann_path  = data_root / f'annotations/{split}.json'
    image_dir = data_root / f'raw/{split}'

    if ann_path.exists():
        with open(ann_path) as f:
            ann = json.load(f)
        n_images      = len(ann['images'])
        n_annotations = len(ann['annotations'])
        n_files       = len(list(image_dir.glob('*.jpg')))
        print(f'{split.upper()}:')
        print(f'  Annotation images : {n_images}')
        print(f'  Annotation boxes  : {n_annotations}')
        print(f'  Downloaded files  : {n_files} .jpg files\n')
    else:
        print(f'{split.upper()}: Annotations file not found at {ann_path}')

## 6. Ground-Truth Bounding Box Inspection

We render 6 randomly selected training images overlaid with their ground-truth bounding box coordinates.

In [ ]:
import json, random, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/DeepMeow/data')
local_data = Path('data')
data_root  = drive_data if (USE_DRIVE and (drive_data / 'annotations/train.json').exists()) else local_data

ann_file  = data_root / 'annotations/train.json'
train_dir = data_root / 'raw/train'

if ann_file.exists():
    with open(ann_file) as f:
        ann_data = json.load(f)

    id_to_anns = {}
    for ann in ann_data['annotations']:
        id_to_anns.setdefault(ann['image_id'], []).append(ann)
    id_to_img = {img['id']: img for img in ann_data['images']}

    valid_ids  = [img_id for img_id in id_to_anns if (train_dir / id_to_img[img_id]['file_name']).exists()]
    sample_ids = random.sample(valid_ids, min(6, len(valid_ids)))

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('COCO Cat Dataset — Ground-Truth Bounding Box Verification', fontsize=14)

    for ax, img_id in zip(axes.flat, sample_ids):
        img_info = id_to_img[img_id]
        img      = Image.open(train_dir / img_info['file_name']).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"ID: {img_id} | {img_info['width']}x{img_info['height']}", fontsize=9)
        for ann in id_to_anns.get(img_id, []):
            x, y, w, h = ann['bbox']
            ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='#FF6B35', facecolor='none'))
            ax.text(x, y - 4, 'cat', color='#FF6B35', fontsize=8, fontweight='bold')

    os.makedirs('results', exist_ok=True)
    plt.tight_layout()
    plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Sample visualization saved to results/sample_images.png')

## 7. Custom CNN Backbone Forward-Pass & Feature Map Verification

We test our custom `Backbone` module with a dummy batch `[B=2, C=3, H=416, W=416]` to ensure
proper channel dimensions and spatial reduction across multi-scale feature maps ($P_3, P_4, P_5$).

In [ ]:
import torch
from src.models.backbone import Backbone

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
backbone = Backbone().to(device)

total_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f'Backbone parameters: {total_params:,}')

dummy = torch.randn(2, 3, 416, 416, device=device)

with torch.no_grad():
    p3, p4, p5 = backbone(dummy)

print(f'\nFeature map shapes:')
print(f'  P3 (small objects) : {tuple(p3.shape)}   <- 52x52 grid (stride 8)')
print(f'  P4 (medium objects): {tuple(p4.shape)}  <- 26x26 grid (stride 16)')
print(f'  P5 (large objects) : {tuple(p5.shape)}  <- 13x13 grid (stride 32)')
print('\nBackbone forward pass OK!')

---
# Week 2 — FPN, Detection Head & Loss Functions

With the backbone verified in Week 1, we now build and verify the remaining detection components:

1. **`src/utils/boxes.py`**: IoU, CIoU, NMS, anchor generation, box encoding/decoding
2. **`src/models/neck.py`**: Feature Pyramid Network (top-down feature enrichment)
3. **`src/models/head.py`**: Detection head producing per-anchor predictions
4. **`src/losses/detection_loss.py`**: CIoU box regression + Focal objectness + BCE classification
5. **`src/models/detector.py`**: Full end-to-end detector (Backbone -> FPN -> Head -> Loss)

## 9. Pull Latest Code from GitHub

In [ ]:
import os

# Pull the latest Week 2 modules from GitHub
!git -C /content/DeepMeow pull

os.chdir('/content/DeepMeow')
print(f'Working directory: {os.getcwd()}')

## 10. Box Utilities Verification (IoU, NMS, Anchors)

We verify the mathematical primitives in `src/utils/boxes.py`.
These are the core operations used throughout the detector pipeline:
- IoU measures how much two boxes overlap (1.0 = identical, 0.0 = no overlap)
- NMS removes duplicate detections by suppressing lower-scoring overlapping boxes
- Anchor generation creates the reference boxes distributed across each feature map grid

In [ ]:
import torch
from src.utils.boxes import compute_iou, compute_ciou, nms, generate_anchors

# IoU test: identical boxes should give IoU = 1.0
box_a = torch.tensor([[10., 10., 50., 50.]])
box_b = torch.tensor([[10., 10., 50., 50.]])
print(f'IoU (identical boxes): {compute_iou(box_a, box_b).item():.4f}  (expect 1.0)')

# IoU test: completely non-overlapping boxes should give IoU = 0.0
box_c = torch.tensor([[100., 100., 150., 150.]])
print(f'IoU (no overlap):      {compute_iou(box_a, box_c).item():.4f}  (expect 0.0)')

# NMS test: box at index 1 overlaps heavily with index 0 and should be suppressed
boxes  = torch.tensor([[10., 10., 50., 50.], [12., 12., 52., 52.], [200., 200., 250., 250.]])
scores = torch.tensor([0.9, 0.8, 0.95])
kept   = nms(boxes, scores, iou_threshold=0.5)
print(f'NMS kept indices: {kept.tolist()}  (expect [2, 0], box 1 suppressed due to overlap with box 0)')

# Anchor generation test: 13x13 grid x 3 anchors = 507 anchors
anchors_p5 = generate_anchors(feature_map_size=13, anchor_sizes=[[116,90],[156,198],[373,326]], stride=32)
print(f'Anchors shape (P5, 13x13, 3 sizes): {anchors_p5.shape}  (expect torch.Size([507, 4]))')

print('\nBox utilities verification passed!')

## 11. Feature Pyramid Network (FPN) Verification

The FPN receives the three backbone feature maps and fuses them via a top-down pathway,
propagating high-level semantic information back to the spatially detailed shallow layers.
After FPN, all three scales have uniform 256-channel outputs enriched with context.

In [ ]:
import torch
from src.models.backbone import Backbone
from src.models.neck import FPN

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
backbone = Backbone().to(device)
fpn      = FPN(out_channels=256).to(device)

dummy = torch.randn(2, 3, 416, 416, device=device)

with torch.no_grad():
    p3, p4, p5 = backbone(dummy)
    f3, f4, f5 = fpn(p3, p4, p5)

print('FPN output shapes (all 256-channel after lateral connection fusion):')
print(f'  F3 (enriched small-scale) : {tuple(f3.shape)}   (expect [2, 256, 52, 52])')
print(f'  F4 (enriched mid-scale)   : {tuple(f4.shape)}  (expect [2, 256, 26, 26])')
print(f'  F5 (enriched large-scale) : {tuple(f5.shape)}  (expect [2, 256, 13, 13])')
print('\nFPN verification passed!')

## 12. Detection Head Verification

The MultiScaleHead produces one prediction tensor per FPN scale.
For each anchor at each grid cell, the head outputs 6 values:
- Indices `[0:4]`: box offset predictions `[tx, ty, tw, th]`
- Index `[4]`: objectness logit (sigmoid -> P(object exists here))
- Index `[5]`: class logit (sigmoid -> P(cat | object))

In [ ]:
from src.models.head import MultiScaleHead

# f3, f4, f5 are already computed in the FPN cell above
head = MultiScaleHead(in_channels=256, num_anchors_per_scale=3, num_classes=1).to(device)

with torch.no_grad():
    pred3, pred4, pred5 = head(f3, f4, f5)

print('Detection head output shapes [B, H, W, num_anchors, pred_per_anchor]:')
print(f'  pred3 : {tuple(pred3.shape)}   (expect [2, 52, 52, 3, 6])')
print(f'  pred4 : {tuple(pred4.shape)}  (expect [2, 26, 26, 3, 6])')
print(f'  pred5 : {tuple(pred5.shape)}  (expect [2, 13, 13, 3, 6])')
print('\nDetection head verification passed!')

## 13. End-to-End Detector: Training Forward Pass & Loss

We test the fully assembled `DeepMeowDetector` in training mode.
Given a batch of images and annotated bounding boxes, the model:
1. Extracts multi-scale features via backbone + FPN
2. Generates predictions at all 3 scales via the detection head
3. Matches anchors to ground-truth boxes using IoU thresholds (pos >= 0.5, neg < 0.4)
4. Computes the three-part loss: CIoU box regression + Focal objectness + BCE classification

In [ ]:
import torch
from src.models.detector import DeepMeowDetector

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = DeepMeowDetector(num_classes=1, input_size=416).to(device)
model.train()

print(f'Total trainable parameters: {model.count_parameters():,}')

# Dummy batch: 2 images, each with cats at known positions
dummy_images = torch.randn(2, 3, 416, 416, device=device)

dummy_targets = [
    {
        'boxes':  torch.tensor([[50., 30., 150., 120.], [200., 100., 300., 250.]], device=device),
        'labels': torch.tensor([0, 0], device=device),
    },
    {
        'boxes':  torch.tensor([[80., 60., 200., 180.]], device=device),
        'labels': torch.tensor([0], device=device),
    },
]

loss, loss_dict = model(dummy_images, dummy_targets)

print(f'\nLoss breakdown (untrained model, values will be high):')
print(f'  Total loss         : {loss_dict["total"]:.4f}')
print(f'  Box (CIoU) loss    : {loss_dict["box"]:.4f}')
print(f'  Objectness loss    : {loss_dict["obj"]:.4f}')
print(f'  Classification loss: {loss_dict["cls"]:.4f}')
print('\nTraining forward pass verified!')

## 14. End-to-End Detector: Inference Mode

In inference mode, the detector decodes box offsets to absolute pixel coordinates
and applies NMS to collapse duplicate detections. Since the model is untrained,
detections are random — this cell only verifies the pipeline is numerically stable
and that the output format (boxes, scores, labels) is correct.

In [ ]:
# Low confidence threshold to see some results even from an untrained model
results = model.predict(dummy_images, conf_threshold=0.01, iou_threshold=0.45)

print('Inference output (untrained model, random predictions):')
for i, r in enumerate(results):
    n_det = r['boxes'].shape[0]
    print(f'  Image {i}: {n_det} detections after NMS')
    if n_det > 0:
        print(f'    Sample box : {r["boxes"][0].tolist()}')
        print(f'    Sample score: {r["scores"][0].item():.4f}')

print('\nInference pipeline verified!')

## 15. Week 2 Milestone Summary

### Completed this week:

| Module | Description | Status |
|--------|-------------|--------|
| `src/utils/boxes.py` | IoU, CIoU, NMS, anchor generation, encode/decode | Done |
| `src/models/neck.py` | Feature Pyramid Network with top-down pathway | Done |
| `src/models/head.py` | Multi-scale anchor prediction head (6 values/anchor) | Done |
| `src/losses/detection_loss.py` | CIoU + Focal + BCE multi-task loss | Done |
| `src/models/detector.py` | End-to-end detector with train/inference modes | Done |

### Week 3 Planned Work:
1. **Training Loop** (`src/train.py`): AdamW optimizer, cosine annealing LR schedule with linear warmup, gradient clipping
2. **Advanced Augmentation**: Mosaic and Mixup data augmentation strategies
3. **mAP Evaluation** (`src/utils/metrics.py`): COCO-style mean Average Precision
4. **First Full Training Run**: 50+ epoch run on the COCO cat dataset to observe initial convergence